# Hito 3 - Explicabilidad del Modelo con SHAP
## GuardianAI - Deteccion Inteligente de Fraude Bancario

## Indice

1. Importacion de librerias
2. Reconstruccion del pipeline
   - 2.1. Carga y particion de datos
   - 2.2. Codificacion, escalado y SMOTE
   - 2.3. Entrenamiento del modelo XGBoost optimizado
3. Que es SHAP y por que se usa
4. Calculo de valores SHAP
5. Interpretacion global: Summary Plot
6. Interpretacion local: Waterfall Plot
7. Contexto de resultados y analisis critico
8. Conclusiones del analisis de explicabilidad

## 1. Importacion de librerias

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

os.makedirs('../figures', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10
})

## 2. Reconstruccion del pipeline

Este notebook es independiente de NB03. Para calcular los valores SHAP necesitamos el modelo entrenado y el conjunto de test preprocesado, por lo que reconstruimos aqui el mismo pipeline exacto de NB03.

El modelo analizado es el **XGBoost optimizado**, que fue el ganador de la comparativa de NB03 con F1=0.1912, PR-AUC=0.1174 y ROC-AUC=0.8730 sobre el test real con desbalanceo original.

### 2.1. Carga y particion de datos

In [ ]:
df = pd.read_csv('Base.csv')
df = df.drop(columns=['credit_risk_score'])

X = df.drop(columns=['fraud_bool'])
y = df['fraud_bool']

# Misma particion 70/15/15 que en NB03
X_train_70, X_temp, y_train_70, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val_15, X_test_15, y_val_15, y_test_15 = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train:      {len(X_train_70):,} registros')
print(f'Validacion: {len(X_val_15):,} registros')
print(f'Test:       {len(X_test_15):,} registros')

### 2.2. Codificacion, escalado y SMOTE

In [ ]:
# One-hot encoding
X_train_enc = pd.get_dummies(X_train_70, drop_first=True)
X_test_enc  = pd.get_dummies(X_test_15,  drop_first=True)
X_test_enc  = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# Solo se escalan las variables continuas, no las columnas OHE
columnas_continuas = [
    'income', 'name_email_similarity', 'prev_address_months_count',
    'current_address_months_count', 'customer_age', 'days_since_request',
    'intended_balcon_amount', 'zip_count_4w', 'velocity_6h',
    'velocity_24h', 'velocity_4w', 'bank_branch_count_8w',
    'date_of_birth_distinct_emails_4w', 'bank_months_count',
    'proposed_credit_limit', 'session_length_in_minutes',
    'device_distinct_emails_8w', 'device_fraud_count', 'month'
]
columnas_continuas = [c for c in columnas_continuas if c in X_train_enc.columns]

scaler = StandardScaler()
scaler.fit(X_train_enc[columnas_continuas])
X_train_enc[columnas_continuas] = scaler.transform(X_train_enc[columnas_continuas])
X_test_enc[columnas_continuas]  = scaler.transform(X_test_enc[columnas_continuas])

# SMOTE solo sobre train
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_enc, y_train_70)

print(f'Train tras SMOTE: {len(X_train_smote):,} muestras '
      f'(fraude: {y_train_smote.mean()*100:.1f}%)')
print(f'Test (desbalanceo real): {len(X_test_enc):,} muestras '
      f'(fraude: {y_test_15.mean()*100:.2f}%)')

### 2.3. Entrenamiento del modelo XGBoost optimizado

Se usan los hiperparametros que encontro `RandomizedSearchCV` en NB03. Estos parametros son los que mejor puntuacion PR-AUC obtuvieron en la validacion cruzada sobre los datos de entrenamiento balanceados con SMOTE.

In [ ]:
# Hiperparametros optimos encontrados en NB03
xgb_model = XGBClassifier(
    subsample=1.0,
    reg_lambda=0.0,
    reg_alpha=0.5,
    n_estimators=250,
    min_child_weight=1,
    max_depth=9,
    learning_rate=0.2,
    colsample_bytree=0.7,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

xgb_model.fit(X_train_smote, y_train_smote)
print('Modelo entrenado correctamente.')
print(f'Features disponibles: {X_train_smote.shape[1]}')

## 3. Que es SHAP y por que se usa

XGBoost combina cientos de arboles de decision para hacer sus predicciones. Eso lo hace muy potente, pero tambien muy dificil de entender: no es posible saber a simple vista por que el modelo decide que una solicitud concreta es un fraude. En el sector bancario esto es un problema serio: la regulacion europea (GDPR) exige que las decisiones automatizadas que afectan a personas sean explicables y auditables.

**SHAP** (SHapley Additive exPlanations) resuelve esto aplicando un concepto de la teoria de juegos llamado **valores de Shapley**. La idea es sencilla: imagina que cada variable del modelo es un jugador en un equipo, y la prediccion final es el resultado del equipo. Los valores de Shapley calculan cuanto ha contribuido cada jugador al resultado, probando todas las combinaciones posibles de jugadores y midiendo cuanto cambia el resultado al incluir o excluir cada uno.

En la practica, cada prediccion se descompone asi:

```
prediccion = valor_base + contribucion(var_1) + contribucion(var_2) + ...
```

donde el `valor_base` es la probabilidad media de fraude en el dataset, y cada contribucion puede ser positiva (empuja hacia fraude) o negativa (empuja hacia legitima).

**Por que SHAP y no la importancia de Gini de NB03:**

- La importancia de Gini dice *cuanto* usa el modelo una variable en promedio, pero no dice *en que direccion* ni *en que casos concretos*.
- SHAP da tanto la magnitud como la direccion de cada variable, tanto a nivel global (comportamiento promedio del modelo) como local (explicacion de una prediccion individual).
- SHAP tiene base matematica solida: es la unica forma de distribuir el credito entre variables que cumple simultaneamente eficiencia, simetria y linealidad — propiedades matematicamente deseables para una explicacion justa.

Para XGBoost se usa `TreeExplainer`, que aprovecha la estructura de los arboles para calcular los valores SHAP exactos de forma eficiente, sin necesidad de aproximaciones.

## 4. Calculo de valores SHAP

Calcular SHAP sobre las 150.000 filas del test completo seria muy lento. Se toma una muestra aleatoria representativa de **1.000 filas** del test, suficiente para que los patrones globales sean estables. Esta practica es estandar cuando el dataset es grande.

El conjunto de test conserva el desbalanceo original (~1.1% de fraudes), por lo que la muestra contiene relativamente pocos fraudes reales. Para los graficos globales eso no es un problema porque SHAP analiza las contribuciones de las variables independientemente de la clase real. Para el waterfall plot se seleccionara especificamente una muestra con probabilidad alta de fraude.

In [ ]:
N_MUESTRA = 1000
np.random.seed(42)
indices_muestra = np.random.choice(len(X_test_enc), size=N_MUESTRA, replace=False)
indices_muestra = np.sort(indices_muestra)

X_shap = X_test_enc.iloc[indices_muestra].reset_index(drop=True)
X_shap = X_shap.astype(np.float32)

print(f'Muestra para SHAP: {X_shap.shape[0]} filas x {X_shap.shape[1]} variables')

# TreeExplainer: version rapida especifica para modelos basados en arboles
explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer.shap_values(X_shap, check_additivity=False)

print('Valores SHAP calculados correctamente.')
print(f'Forma de la matriz SHAP: {shap_vals.shape}')
print('(filas = muestras, columnas = variables, '
      'valores = contribucion a la prediccion)')

## 5. Interpretacion global: Summary Plot

El summary plot muestra el comportamiento global del modelo: que variables usa mas y en que direccion.

**Como leer este grafico:**

- Las variables aparecen ordenadas de mayor a menor importancia media (la variable que mas influye en las predicciones va arriba).
- Cada punto representa una muestra del test. Su posicion horizontal indica el valor SHAP de esa variable para esa muestra: positivo significa que esa variable empuja la prediccion hacia fraude, negativo significa que la empuja hacia legitima.
- El color del punto indica el valor real de la variable en esa muestra: rojo = valor alto, azul = valor bajo.
- Una nube de puntos ancha indica que el impacto de esa variable varia mucho segun el caso concreto.

Por ejemplo: si `keep_alive_session` aparece arriba y sus puntos rojos estan a la derecha, significa que tener la sesion activa (valor alto) aumenta la probabilidad de fraude segun el modelo.

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals, X_shap,
    max_display=20,
    show=False
)
plt.title('Importancia global de variables (SHAP) — Top 20',
          fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../figures/shap_summary_dot.png', dpi=150, bbox_inches='tight')
plt.show()

El grafico de puntos anterior muestra la direccion de cada variable. A continuacion, el grafico de barras resume la importancia media absoluta de cada variable, mas sencillo de leer para comparar magnitudes:

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_vals, X_shap,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title('Importancia media de variables (|valor SHAP| medio) — Top 20',
          fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../figures/shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Interpretacion local: Waterfall Plot

El summary plot explica el comportamiento promedio del modelo. El waterfall plot explica **una prediccion individual concreta**: por que el modelo decidio que esa solicitud especifica era un fraude.

Esto es especialmente util en banca: si un cliente reclama que se le ha bloqueado injustamente, el analista puede ver exactamente que variables activaron la alerta y con que intensidad.

**Como leer este grafico:**

- La barra arranca en el `valor base` (probabilidad media de fraude en el dataset completo).
- Cada variable suma o resta su contribucion SHAP a ese valor base.
- Las barras rojas empujan hacia fraude (contribucion positiva).
- Las barras azules empujan hacia legitima (contribucion negativa).
- El resultado final es la probabilidad predicha para ese caso concreto.

Se selecciona la muestra con mayor probabilidad de fraude predicha para que la explicacion sea mas ilustrativa.

In [ ]:
# Predicciones sobre la muestra
y_prob_muestra = xgb_model.predict_proba(X_shap)[:, 1]

# Usar la muestra con mayor probabilidad de fraude
idx_local = int(np.argmax(y_prob_muestra))

print(f'Muestra seleccionada: indice {idx_local}')
print(f'Probabilidad de fraude predicha: {y_prob_muestra[idx_local]:.4f}')

In [ ]:
# Calcular la explicacion SHAP para esa muestra concreta
explicacion = explainer(X_shap, check_additivity=False)

plt.figure()
shap.plots.waterfall(explicacion[idx_local], max_display=15, show=False)
plt.title(
    f'Explicacion local — muestra {idx_local} '
    f'(prob. fraude: {y_prob_muestra[idx_local]:.4f})',
    fontsize=11, fontweight='bold', pad=12
)
plt.tight_layout()
plt.savefig('../figures/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Contexto de resultados y analisis critico

### Resultados globales del proyecto

Antes de interpretar SHAP es importante situar el rendimiento real del modelo en el contexto de todo el trabajo realizado:

| Notebook | Modelo | F1 | PR-AUC | ROC-AUC | Nota |
|----------|--------|-----|--------|---------|------|
| NB02 | XGBoost (sin SMOTE) | **0.2359** | **0.1711** | **0.8944** | Mejor resultado global |
| NB02 | Reg. Logistica | 0.2110 | 0.1363 | 0.8755 | — |
| NB02 | GNN | 0.2060 | 0.1333 | 0.8710 | 1527s de entrenamiento |
| NB02 | Random Forest | 0.1956 | 0.1243 | 0.8768 | — |
| NB02 | LSTM | 0.1183 | 0.0593 | 0.7998 | — |
| NB03 | XGBoost opt. + SMOTE | 0.1912 | 0.1174 | 0.8730 | Modelo analizado aqui |
| NB03 | RF opt. + SMOTE | 0.1652 | 0.0910 | 0.8551 | 142 min de busqueda |
| NB03 | LR opt. + SMOTE | 0.1653 | 0.0816 | 0.8316 | — |

**Observacion clave**: el mejor resultado del proyecto no proviene del modelo con SMOTE y optimizacion, sino del XGBoost simple de NB02 sin ninguno de esos tratamientos. SMOTE empeoro los resultados en este dataset concreto.

### Por que SMOTE no mejoro

Bank Account Fraud (NeurIPS 2022) es un dataset **sintetico**: sus datos fueron generados artificialmente a partir de patrones de datos bancarios reales, no son transacciones bancarias reales. Cuando SMOTE genera muestras sinteticas interpolando entre muestras ya sinteticas, introduce una doble artificialidad que anade ruido en lugar de informacion util. En datasets con datos reales, SMOTE habitualmente aporta mejoras claras.

Este resultado es un hallazgo valido del proyecto: demuestra que SMOTE no es una solucion universal y que su eficacia depende de la naturaleza de los datos.

### Por que los resultados absolutos son bajos

Detectar entre el 20% y el 25% de los fraudes con una precision del 18-24% puede parecer un mal resultado, pero es lo esperable con datos tabulares sinteticos sin informacion temporal. Los sistemas de deteccion de fraude en produccion real combinan:

- **Datos temporales**: secuencias de transacciones, velocidades de cambio en el comportamiento del usuario a lo largo del tiempo.
- **Grafos de relaciones**: conexiones entre cuentas, dispositivos y geolocalizaciones que permiten detectar redes de fraude organizadas.
- **Calibracion del umbral al coste de negocio**: en banca, bloquear un fraude de 5.000 euros puede justificar generar hasta 20 falsas alarmas si el coste de cada falsa alarma (gestion manual) es de 200 euros.
- **Retroalimentacion continua**: el modelo se reentrena frecuentemente con los fraudes confirmados mas recientes.

Nada de esto es posible con este dataset. El valor del trabajo es metodologico, no en los numeros absolutos.

### Sobre los valores SHAP con este modelo

SHAP explica *lo que hace el modelo*, no *lo que deberia hacer*. Las variables que aparecen como mas importantes son aquellas en las que el modelo se apoya para tomar sus decisiones — correctas o no. Con un F1 de 0.19, el modelo toma muchas decisiones incorrectas, y SHAP nos muestra los patrones de esas decisiones, incluyendo los errores.

Dicho esto, si los patrones que muestra SHAP tienen sentido desde el punto de vista del negocio bancario (las variables mas importantes son realmente indicadores razonables de fraude), eso es una validacion cualitativa de que el modelo ha aprendido algo real, aunque le quede mucho margen de mejora.

## 8. Conclusiones del analisis de explicabilidad

El analisis SHAP cumple dos funciones en el sistema GuardianAI:

**A nivel global** (summary plot), permite identificar que variables son las mas influyentes y en que direccion actuan. Esto sirve para validar cualitativamente que el modelo ha aprendido patrones con sentido de negocio: si las variables mas importantes son indicadores logicos de comportamiento fraudulento, el modelo esta aprendiendo algo real aunque sus metricas cuantitativas sean modestas.

**A nivel local** (waterfall plot), permite explicar cualquier prediccion individual de forma trazable. Esto satisface dos requisitos criticos:

- **Regulatorio**: el GDPR exige que las decisiones automatizadas que afectan a personas sean explicables y auditables. Un sistema que solo devuelve una probabilidad sin explicacion no cumple este requisito.
- **Operativo**: los analistas de riesgo pueden revisar los casos limite con contexto real, en lugar de basarse ciegamente en un numero.

### Limitaciones identificadas

1. El modelo sobre el que se aplica SHAP (XGBoost optimizado con SMOTE) no es el mejor del proyecto: el XGBoost de NB02 sin SMOTE obtuvo F1=0.2359 frente al F1=0.1912 de este. Si el objetivo fuera maximizar el rendimiento del sistema de produccion, se usaria el modelo de NB02. Se mantiene el de NB03 porque es metodologicamente mas completo (incluye preprocesado, optimizacion y SMOTE).

2. SHAP explica el modelo actual. Si en iteraciones futuras el modelo mejora sustancialmente (por ingenieria de caracteristicas, datos temporales o arquitecturas mas sofisticadas), el analisis SHAP cambiara.

3. La muestra de 1.000 casos es representativa estadisticamente, pero los patrones del summary plot pueden variar ligeramente con muestras distintas. Para un analisis en produccion se usarian al menos 5.000-10.000 muestras.

### Aplicacion en el Hito 4 (despliegue con FastAPI)

La integracion natural de SHAP en produccion seria devolver, junto a cada prediccion de la API, las 3-5 variables que mas influyeron en esa decision concreta y en que sentido. Por ejemplo:

```json
{
  "probabilidad_fraude": 0.73,
  "decision": "ALERTA",
  "explicacion": [
    {"variable": "keep_alive_session", "direccion": "fraude", "peso": 0.31},
    {"variable": "velocity_6h",        "direccion": "fraude", "peso": 0.18},
    {"variable": "income",              "direccion": "legitima","peso": -0.09}
  ]
}
```

Eso convierte el sistema en algo auditable y util en la practica, independientemente de las limitaciones actuales de las metricas.